# 260501 promotion split

`Membership_v1.csv`의 `is_promotion` 값을 기준으로 데이터를 2개 집단으로 분리한다.

- `promotion_1`: `is_promotion == 1`인 멤버십 행에 등장한 `USER_KEY` 기준
- `promotion_0`: `is_promotion == 0`인 멤버십 행에 등장한 `USER_KEY` 기준
- `View_History_v1.csv`는 `User_Mapping_v1.csv`를 거쳐 `USER_NUM` 기준으로 필터링한다.
- `Movie_Master_v1.csv`는 각 집단의 시청 이력에 등장한 `MOVIE_NUM`만 남긴다.


In [ ]:
from pathlib import Path

import pandas as pd


BASE_DIR = Path(r"_data/02_interim")

if not BASE_DIR.exists():
    BASE_DIR = Path.cwd().parent

SOURCE_DIR = next(BASE_DIR.glob("260430*"))
OUTPUT_DIR = BASE_DIR / "260501_promotion_split"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MEMBERSHIP_PATH = SOURCE_DIR / "Membership_v1.csv"
USER_MAPPING_PATH = SOURCE_DIR / "User_Mapping_v1.csv"
VIEW_HISTORY_PATH = SOURCE_DIR / "View_History_v1.csv"
MOVIE_MASTER_PATH = SOURCE_DIR / "Movie_Master_v1.csv"

membership = pd.read_csv(MEMBERSHIP_PATH)
user_mapping = pd.read_csv(USER_MAPPING_PATH)
view_history = pd.read_csv(VIEW_HISTORY_PATH)
movie_master = pd.read_csv(MOVIE_MASTER_PATH)

print("membership:", membership.shape)
print("user_mapping:", user_mapping.shape)
print("view_history:", view_history.shape)
print("movie_master:", movie_master.shape)


In [ ]:
required_columns = {
    "membership": {"USER_KEY", "is_promotion"},
    "user_mapping": {"USER_KEY", "USER_NUM"},
    "view_history": {"USER_NUM", "MOVIE_NUM"},
    "movie_master": {"MOVIE_NUM"},
}

dataframes = {
    "membership": membership,
    "user_mapping": user_mapping,
    "view_history": view_history,
    "movie_master": movie_master,
}

for name, columns in required_columns.items():
    missing_columns = columns - set(dataframes[name].columns)
    if missing_columns:
        raise ValueError(f"{name}에 필요한 컬럼이 없습니다: {sorted(missing_columns)}")

print("필수 컬럼 확인 완료")


In [ ]:
def split_by_promotion(promotion_value: int) -> dict[str, pd.DataFrame]:
    membership_split = membership.loc[
        membership["is_promotion"].eq(promotion_value)
    ].copy()

    user_keys = membership_split["USER_KEY"].dropna().unique()

    user_mapping_split = user_mapping.loc[
        user_mapping["USER_KEY"].isin(user_keys)
    ].copy()

    user_nums = user_mapping_split["USER_NUM"].dropna().unique()

    view_history_split = view_history.loc[
        view_history["USER_NUM"].isin(user_nums)
    ].copy()

    movie_nums = view_history_split["MOVIE_NUM"].dropna().unique()

    movie_master_split = movie_master.loc[
        movie_master["MOVIE_NUM"].isin(movie_nums)
    ].copy()

    return {
        "membership": membership_split,
        "user_mapping": user_mapping_split,
        "view_history": view_history_split,
        "movie_master": movie_master_split,
    }


promotion_1 = split_by_promotion(1)
promotion_0 = split_by_promotion(0)

summary = []

for group_name, group_data in {
    "promotion_1": promotion_1,
    "promotion_0": promotion_0,
}.items():
    for data_name, data in group_data.items():
        summary.append(
            {
                "group": group_name,
                "data": data_name,
                "rows": len(data),
                "columns": len(data.columns),
            }
        )

pd.DataFrame(summary)


In [ ]:
output_map = {
    "promotion_1_membership.csv": promotion_1["membership"],
    "promotion_1_user_mapping.csv": promotion_1["user_mapping"],
    "promotion_1_view_history.csv": promotion_1["view_history"],
    "promotion_1_movie_master.csv": promotion_1["movie_master"],
    "promotion_0_membership.csv": promotion_0["membership"],
    "promotion_0_user_mapping.csv": promotion_0["user_mapping"],
    "promotion_0_view_history.csv": promotion_0["view_history"],
    "promotion_0_movie_master.csv": promotion_0["movie_master"],
}

for file_name, data in output_map.items():
    output_path = OUTPUT_DIR / file_name
    data.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"saved: {output_path} / rows={len(data):,}")


In [ ]:
promotion_1_user_keys = set(promotion_1["membership"]["USER_KEY"])
promotion_0_user_keys = set(promotion_0["membership"]["USER_KEY"])
overlap_user_keys = promotion_1_user_keys & promotion_0_user_keys

print("promotion_1 USER_KEY 수:", len(promotion_1_user_keys))
print("promotion_0 USER_KEY 수:", len(promotion_0_user_keys))
print("두 집단에 모두 등장하는 USER_KEY 수:", len(overlap_user_keys))
